# i2 — Evidence-Gated Sparse Forensic Network v1

Self-contained implementation of the conservative MoE design in `specialArchitecture.md` on the lightweight-generalization branch. Matching architecture: [i2 specification](../architecture/i2-Evidence-Gated-Sparse-Forensic-v1.md). **Prepared, not executed or benchmarked.** No i2 Pass/Fail record exists yet.

Run configuration → definitions → preflight → data audit → training/calibration. Final testing is a separate explicit opt-in. Five epochs are a smoke experiment, not a performance claim. Pre-extracted aligned face crops are required; face detection and identity inference are outside this notebook.

Dependencies match i1: torch, torchvision, numpy, Pillow, scikit-learn. Training requires cached/downloadable torchvision EfficientNet-B0 weights. Preflight uses random weights and does not download. No silent random-weight training fallback.

## Required manifest
Set `MANIFEST_PATH` to a JSON list of records. Required fields: `split` (train/dev/calibration/test), `dataset`, `video_id`, `frames_dir`, integer `label` (0/1), nonempty `identity_ids` and `source_ids` lists. IDs must be globally canonical and include all source/target identities and video ancestors. Optional `method`, `masks_dir` (PNG masks matching frame stems). Relative directories resolve against the manifest file. Frame filenames must sort chronologically. See the architecture for the complete contract.

Do not fabricate identity IDs from filenames just to pass validation. Training rows must be real. With `SYNTHETIC_DEV_CAL=True`, dev/calibration rows must also be real and receive paired synthetic examples; resulting calibration is synthetic-only. Set it false only for separately curated actual-real/fake dev/calibration splits. Test always uses actual labels; the unseen dataset stays test-only. All four splits are required.

Outputs go to a new `OUT_DIR`: configuration, manifest, history, best/last checkpoints, calibration, and optionally final test scores/metrics. No outputs are fabricated or committed in this notebook.

In [ ]:
import os, json, math, random, hashlib, time, copy, io
from pathlib import Path
from dataclasses import dataclass, asdict
from contextlib import nullcontext
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance, ImageDraw
import PIL
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import sklearn
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix, accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score

@dataclass
class Config:
    name: str = 'i2-Evidence-Gated-Sparse-Forensic-v1'
    manifest_path: str = os.environ.get('I2_MANIFEST', '/kaggle/working/i2_manifest.json')
    out_dir: str = os.environ.get('I2_OUT_DIR', '/kaggle/working/outputs/i2-egsf-v1-A6-seed0')
    seed: int = 0
    ablation: int = 6  # 0 shared; 1 residual; 2 temporal; 3 localization; 4 MoE; 5 DCT; 6 Q/U
    image_size: int = 224
    frames: int = 12
    batch_size: int = 2
    accumulation: int = 2
    workers: int = 2
    epochs: int = 5
    warmup_epochs: int = 1
    head_lr: float = 5e-4
    backbone_lr: float = 1e-4
    weight_decay: float = 1e-4
    capacity_factor: float = 1.25
    expert_dropout: float = 0.05
    synthetic_dev_cal: bool = True
    unseen_dataset: str = 'dfdcp'
    max_fpr: float = 0.01
    max_false_clear: float = 0.05
    min_fake_recall: float = 0.50
    bootstrap: int = 1000

CFG = Config()
MANIFEST_PATH = Path(CFG.manifest_path)
OUT_DIR = Path(CFG.out_dir)
RUN_TRAIN = True
RUN_FINAL_TEST = False  # Enable only after training and calibration are frozen.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CONDITIONS = ('clean', 'jpeg30', 'blur15', 'resize50', 'lowres25')

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def amp_context():
    return torch.autocast('cuda', dtype=torch.float16) if DEVICE.type == 'cuda' else nullcontext()

def write_json(path, obj):
    def convert(x):
        if isinstance(x, dict): return {str(k): convert(v) for k, v in x.items()}
        if isinstance(x, (list, tuple)): return [convert(v) for v in x]
        if isinstance(x, np.ndarray): return convert(x.tolist())
        if isinstance(x, np.generic): return convert(x.item())
        if isinstance(x, float) and not math.isfinite(x): return None
        return x
    path = Path(path); temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(convert(obj), indent=2, allow_nan=False))
    temp.replace(path)

VERSIONS = dict(torch=torch.__version__, torchvision=torchvision.__version__, numpy=np.__version__, pillow=PIL.__version__, sklearn=sklearn.__version__, device=str(DEVICE))
seed_all(CFG.seed)
assert CFG.image_size == 224 and 0 <= CFG.ablation <= 6
assert CFG.frames > 0 and CFG.batch_size > 0 and CFG.accumulation > 0 and CFG.epochs > 0
assert 0 < CFG.max_fpr < 1 and 0 < CFG.max_false_clear < 1 and 0 < CFG.min_fake_recall <= 1
assert CFG.capacity_factor > 0 and 0 <= CFG.expert_dropout < 1
print(VERSIONS)
print(asdict(CFG))


In [ ]:
# Metadata validation never loads test image pixels during training.
def read_manifest(path, cfg):
    path = Path(path).resolve()
    if not path.is_file():
        raise FileNotFoundError('Set MANIFEST_PATH/CFG.manifest_path to an identity/ancestry-audited JSON manifest; see the first cell. Missing: ' + str(path))
    rows = json.loads(path.read_text())
    if not isinstance(rows, list) or not rows:
        raise ValueError('Manifest must be a nonempty list of video records.')
    required = {'split', 'dataset', 'video_id', 'frames_dir', 'label', 'identity_ids', 'source_ids'}
    seen_video, seen_paths, owners, parent = set(), set(), {}, {}
    def root(k):
        parent.setdefault(k, k)
        if parent[k] != k: parent[k] = root(parent[k])
        return parent[k]
    for row in rows:
        if not required.issubset(row): raise ValueError('Missing manifest fields: ' + str(required - set(row)))
        if row['split'] not in ('train', 'dev', 'calibration', 'test'): raise ValueError('Invalid split')
        if type(row['label']) is not int or row['label'] not in (0, 1): raise ValueError('label must be integer 0 or 1')
        if not isinstance(row['dataset'], str) or not row['dataset'].strip(): raise ValueError('dataset must be nonempty')
        row['dataset'] = row['dataset'].casefold()
        key = (row['dataset'], str(row['video_id']))
        if key in seen_video: raise ValueError('Repeated dataset/video_id: ' + str(key))
        seen_video.add(key)
        fp = Path(row['frames_dir']); fp = fp if fp.is_absolute() else path.parent / fp
        fp = fp.resolve()
        if str(fp) in seen_paths: raise ValueError('Reused frames directory: ' + str(fp))
        seen_paths.add(str(fp)); row['frames_dir'] = str(fp)
        frames = sorted(p for p in fp.glob('*') if p.suffix.lower() in ('.png', '.jpg', '.jpeg'))
        if not frames: raise ValueError('No frames: ' + str(fp))
        row['frames'] = [str(p) for p in frames]
        if row.get('masks_dir'):
            mp = Path(row['masks_dir']); mp = mp if mp.is_absolute() else path.parent / mp
            row['masks_dir'] = str(mp.resolve())
            if any(not (mp / (p.stem + '.png')).is_file() for p in frames): raise ValueError('Missing region masks: ' + str(mp))
        keys = []
        for field in ('identity_ids', 'source_ids'):
            ids = row[field]
            if not isinstance(ids, list) or not ids or any(not isinstance(v, str) or not v.strip() for v in ids):
                raise ValueError(field + ' must contain canonical nonempty strings')
            for v in ids:
                token = field + ':' + v
                if token in owners and owners[token] != row['split']: raise ValueError('Cross-split identity/ancestry: ' + token)
                owners[token] = row['split']; keys.append(token)
        for k in keys[1:]: parent[root(k)] = root(keys[0])
        row['_group_key'] = keys[0]
        row['method'] = str(row.get('method', 'real' if row['label'] == 0 else 'unknown'))
    for row in rows:
        row['group'] = hashlib.sha256(root(row.pop('_group_key')).encode()).hexdigest()[:16]
    for split in ('train', 'dev', 'calibration', 'test'):
        subset = [r for r in rows if r['split'] == split]
        if not subset: raise ValueError('Empty split: ' + split)
        synthetic = split == 'train' or (cfg.synthetic_dev_cal and split in ('dev', 'calibration'))
        if synthetic and any(r['label'] for r in subset): raise ValueError(split + ' must contain only real source rows in this protocol')
        if not synthetic:
            for d in {r['dataset'] for r in subset}:
                if {r['label'] for r in subset if r['dataset'] == d} != {0, 1}: raise ValueError(split + '/' + d + ' needs both labels')
    unseen = [r for r in rows if r['dataset'] == cfg.unseen_dataset.casefold()]
    if not unseen or any(r['split'] != 'test' for r in unseen): raise ValueError('Configured unseen dataset must exist exclusively in test')
    return rows, hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()

def degrade(img, condition):
    if condition == 'clean': return img.copy()
    if condition == 'jpeg30':
        b = io.BytesIO(); img.save(b, format='JPEG', quality=30); b.seek(0)
        return Image.open(b).convert('RGB').copy()
    if condition == 'blur15': return img.filter(ImageFilter.GaussianBlur(1.5))
    if condition in ('resize50', 'lowres25'):
        scale = 0.5 if condition == 'resize50' else 0.25
        return img.resize((max(1, int(img.width * scale)), max(1, int(img.height * scale))), Image.Resampling.BILINEAR).resize(img.size, Image.Resampling.BILINEAR)
    raise ValueError(condition)

def tensor_image(img):
    return torch.from_numpy(np.array(img, dtype=np.float32, copy=True)).permute(2, 0, 1) / 255.0

def make_blend(img, seed):
    rng = random.Random(seed); size = img.width
    mask = Image.new('L', img.size, 0)
    x0, y0 = int(size * rng.uniform(.12, .22)), int(size * rng.uniform(.08, .20))
    x1, y1 = int(size * rng.uniform(.78, .9)), int(size * rng.uniform(.8, .94))
    ImageDraw.Draw(mask).ellipse((x0, y0, x1, y1), fill=255)
    region = rng.choice(('full', 'upper', 'lower', 'middle'))
    a = np.asarray(mask, dtype=np.float32).copy()
    if region == 'upper': a[int(size * .6):] = 0
    if region == 'lower': a[:int(size * .4)] = 0
    if region == 'middle': a[:int(size * .3)] = 0; a[int(size * .7):] = 0
    mask = Image.fromarray(a.astype(np.uint8)).filter(ImageFilter.GaussianBlur(rng.uniform(2, 8)))
    coverage = np.asarray(mask, dtype=np.float32) / 255.0
    alpha = coverage * rng.uniform(.3, .9)
    src = ImageEnhance.Color(img).enhance(rng.uniform(.65, 1.35))
    src = ImageEnhance.Brightness(src).enhance(rng.uniform(.8, 1.2))
    scale = rng.uniform(.7, 1.0)
    src = src.resize((int(size * scale), int(size * scale)), Image.Resampling.BILINEAR).resize(img.size, Image.Resampling.BILINEAR)
    # Reflect padding avoids black-border shortcuts in the shifted source.
    arr = np.asarray(src, dtype=np.float32); pad = 6
    arr = np.pad(arr, ((pad, pad), (pad, pad), (0, 0)), mode='reflect')
    dx, dy = rng.randint(-5, 5), rng.randint(-5, 5)
    arr = arr[pad + dy:pad + dy + size, pad + dx:pad + dx + size]
    blended = arr * alpha[..., None] + np.asarray(img, dtype=np.float32) * (1 - alpha[..., None])
    return Image.fromarray(np.clip(blended, 0, 255).astype(np.uint8)), torch.from_numpy(coverage.copy())

class ClipDataset(Dataset):
    def __init__(self, rows, split, cfg, condition='clean'):
        self.cfg, self.split, self.condition = cfg, split, condition
        self.training = split == 'train'
        self.synthetic = self.training or (cfg.synthetic_dev_cal and split in ('dev', 'calibration'))
        self.samples = []
        for row in rows:
            if row['split'] == split:
                for fake in ((False, True) if self.synthetic else (False,)):
                    self.samples.append((row, fake))
    def __len__(self): return len(self.samples)
    def __getitem__(self, index):
        row, synthetic_fake = self.samples[index]
        seed = random.randrange(2 ** 31) if self.training else self.cfg.seed + index * 100003
        rng = random.Random(seed); files = row['frames']; n = min(self.cfg.frames, len(files))
        if self.training:
            start = rng.randrange(len(files) - n + 1); chosen = list(range(start, start + n))
        else:
            chosen = np.linspace(0, len(files) - 1, n).round().astype(int).tolist()
        assert len(set(chosen)) == n
        x, xi, masks, known = [], [], [], []
        train_condition = rng.choice(('clean', 'clean', 'jpeg30', 'blur15', 'resize50'))
        pair_condition = rng.choice(('blur15', 'resize50'))
        brightness = rng.uniform(.9, 1.1)
        for j in chosen:
            with Image.open(files[j]) as opened: img = opened.convert('RGB').resize((224, 224), Image.Resampling.BILINEAR)
            if synthetic_fake:
                img, mask = make_blend(img, seed)
                has_mask = True
            elif row.get('masks_dir'):
                mp = Path(row['masks_dir']) / (Path(files[j]).stem + '.png')
                with Image.open(mp) as opened: ma = opened.convert('L').resize((224, 224), Image.Resampling.NEAREST)
                mask = torch.from_numpy(np.array(ma, dtype=np.float32, copy=True)) / 255.0; has_mask = True
            else:
                mask = torch.zeros(224, 224); has_mask = row['label'] == 0
            if self.training:
                img = ImageEnhance.Brightness(img).enhance(brightness)
                img = degrade(img, train_condition)
            else: img = degrade(img, self.condition)
            x.append(tensor_image(img))
            xi.append(tensor_image(degrade(img, pair_condition)) if self.training else x[-1])
            masks.append(mask); known.append(has_mask)
        valid = [True] * n; indices = list(chosen)
        while len(x) < self.cfg.frames:
            x.append(torch.zeros_like(x[0])); xi.append(torch.zeros_like(xi[0])); masks.append(torch.zeros_like(masks[0]))
            known.append(False); valid.append(False); indices.append(-100000)
        return dict(x=torch.stack(x), xi=torch.stack(xi), mask=torch.stack(masks), known=torch.tensor(known), valid=torch.tensor(valid), indices=torch.tensor(indices), y=torch.tensor(float(synthetic_fake) if self.synthetic else float(row['label'])), video_id=str(row['video_id']), dataset=row['dataset'], method='synthetic' if synthetic_fake else row['method'], group=row['group'])

def worker_seed(worker):
    seed = torch.initial_seed() % (2 ** 32); random.seed(seed); np.random.seed(seed)

def make_loader(ds, cfg):
    generator = torch.Generator().manual_seed(cfg.seed)
    sampler = None
    if ds.training:
        counts = {}
        for row, _ in ds.samples: counts[row['dataset']] = counts.get(row['dataset'], 0) + 1
        sampler = WeightedRandomSampler([1.0 / counts[r['dataset']] for r, _ in ds.samples], len(ds), replacement=True, generator=generator)
    return DataLoader(ds, batch_size=cfg.batch_size, sampler=sampler, num_workers=cfg.workers, pin_memory=DEVICE.type == 'cuda', worker_init_fn=worker_seed, generator=generator, persistent_workers=False)

def move_batch(batch):
    return {k: v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v for k, v in batch.items()}


In [ ]:
class FixedForensics(nn.Module):
    def __init__(self):
        super().__init__()
        kernels = torch.tensor([[[-1,0,1],[-2,0,2],[-1,0,1]], [[-1,-2,-1],[0,0,0],[1,2,1]], [[0,1,0],[1,-4,1],[0,1,0]]], dtype=torch.float32)
        kernels /= kernels.abs().sum((1,2), keepdim=True)
        self.register_buffer('highpass', kernels[:, None])
        q = torch.arange(8).float(); u = q[:, None]
        d = torch.cos(math.pi * (2*q[None]+1) * u / 16) * math.sqrt(2/8)
        d[0] /= math.sqrt(2)
        self.register_buffer('dct', torch.einsum('ui,vj->uvij', d, d).reshape(64, 1, 8, 8))
        band = (q[:,None] + q[None]).flatten()
        self.register_buffer('low', ((band > 0) & (band <= 2)))
        self.register_buffer('high', band >= 8)
    def forward(self, x, frequency):
        # FP32 protects small forensic energies during AMP training.
        with torch.autocast(x.device.type, enabled=False):
            x = x.float(); y = .299*x[:,0:1] + .587*x[:,1:2] + .114*x[:,2:3]
            hp = F.conv2d(F.pad(y, (1,1,1,1), mode='reflect'), self.highpass.float())
            chroma = torch.cat([x[:,0:1]-y, x[:,2:3]-y], 1)
            chroma = chroma - F.avg_pool2d(F.pad(chroma, (1,1,1,1), mode='reflect'), 3, stride=1)
            if frequency:
                energy = F.conv2d(y, self.dct.float(), stride=8).square()
                lo = energy[:,self.low].mean(1, keepdim=True); hi = energy[:,self.high].mean(1, keepdim=True)
                spectral = torch.cat([torch.log1p(lo), torch.log1p(hi), hi/(lo+hi+1e-6)], 1)
                spectral = F.adaptive_avg_pool2d(spectral, (7,7))
            else: spectral = x.new_zeros(x.shape[0],3,7,7)
            return torch.cat([F.adaptive_avg_pool2d(hp,(7,7)), spectral, F.adaptive_avg_pool2d(chroma,(7,7))], 1)

class SparseExperts(nn.Module):
    def __init__(self, cfg, width=256):
        super().__init__(); self.cfg = cfg
        self.router = nn.Linear(width, 4)
        self.experts = nn.ModuleList([nn.Sequential(nn.Linear(width,128), nn.GELU(), nn.Linear(128,width)) for _ in range(4)])
        self.gate = nn.Parameter(torch.tensor(0.0))
    def forward(self, h, frame_valid):
        normalized = F.layer_norm(h, (h.shape[-1],))
        logits = self.router(normalized)
        probabilities = logits.float().softmax(-1)
        confidence, route = probabilities.max(-1)
        cap = min(h.shape[1], math.ceil(self.cfg.capacity_factor*h.shape[1]/4))
        correction = torch.zeros_like(h); accepted = torch.zeros(4, device=h.device)
        eligible = frame_valid[:,None].expand_as(route)
        requested = (F.one_hot(route,4).float()*eligible[...,None]).sum((0,1))
        capacity_accepted = h.new_zeros(())
        for e, expert in enumerate(self.experts):
            score = confidence.masked_fill((route != e) | ~eligible, -torch.inf)
            values, pos = score.topk(cap, dim=1)
            keep = torch.isfinite(values); capacity_accepted = capacity_accepted + keep.sum()
            if self.training: keep = keep & (torch.rand_like(values) >= self.cfg.expert_dropout)
            f, slot = keep.nonzero(as_tuple=True); token = pos[f,slot]
            if f.numel():
                update = torch.tanh(expert(normalized[f,token]))
                correction[f,token] = (update * probabilities[f,token,e,None] * (.30*torch.sigmoid(self.gate))).to(h.dtype)
            accepted[e] = f.numel()
        count = eligible.sum().clamp_min(1)
        hard = requested.detach() / count
        mean_p = (probabilities * eligible[...,None]).sum((0,1)) / count
        balance = 4*(hard*mean_p).sum()
        entropy = (-(probabilities*probabilities.clamp_min(1e-8).log()).sum(-1)*eligible).sum()/count
        stats = dict(requested=requested.detach(), accepted=accepted.detach(), overflow=(eligible.sum()-capacity_accepted).detach(), tokens=eligible.sum().detach(), entropy=entropy.detach())
        return h + correction, balance, stats

def near_pairs(valid, indices):
    return valid[:,1:] & valid[:,:-1] & ((indices[:,1:]-indices[:,:-1]) > 0) & ((indices[:,1:]-indices[:,:-1]) <= 2)

class ForensicNetwork(nn.Module):
    def __init__(self, cfg, pretrained=True):
        super().__init__(); self.cfg = cfg; self.level = cfg.ablation
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.backbone = efficientnet_b0(weights=weights).features
        self.fixed = FixedForensics()
        self.register_buffer('mean', torch.tensor([.485,.456,.406])[None,:,None,None])
        self.register_buffer('std', torch.tensor([.229,.224,.225])[None,:,None,None])
        self.rgb = nn.Linear(1280,192); self.forensic_proj = nn.Linear(8,32); self.delta = nn.Linear(192,32,bias=False)
        self.norm = nn.LayerNorm(256)
        self.adapter = nn.Sequential(nn.Linear(256,256),nn.GELU(),nn.Dropout(.1),nn.Linear(256,256))
        self.shared = nn.Sequential(nn.LayerNorm(256),nn.Linear(256,256),nn.GELU(),nn.Linear(256,256))
        self.moe = SparseExperts(cfg)
        self.m = nn.Linear(256,1); self.q = nn.Linear(256,1); self.u = nn.Linear(256,1)
        self.temporal_gate = nn.Conv1d(512,1,1)
        self.classifier = nn.Sequential(nn.LayerNorm(256),nn.Linear(256,128),nn.GELU(),nn.Dropout(.1),nn.Linear(128,1))
        self.local_classifier = nn.Linear(256,1)
        self.backbone.requires_grad_(False)
        if self.level < 1: self.forensic_proj.requires_grad_(False)
        if self.level < 2: self.delta.requires_grad_(False); self.temporal_gate.requires_grad_(False)
        if self.level < 3: self.m.requires_grad_(False); self.local_classifier.requires_grad_(False)
        if self.level < 4: self.moe.requires_grad_(False)
        if self.level < 6: self.q.requires_grad_(False); self.u.requires_grad_(False)
    def train(self, mode=True):
        super().train(mode)
        for module in self.backbone.modules():
            if isinstance(module, nn.BatchNorm2d): module.eval()
        return self
    def set_epoch(self, epoch):
        self.backbone.requires_grad_(False)
        if epoch >= self.cfg.warmup_epochs:
            for block in list(self.backbone.children())[-3:]: block.requires_grad_(True)
    def forward(self, x, valid, indices):
        if x.ndim != 5 or x.shape[2:] != (3,224,224): raise ValueError('Expected B,T,3,224,224 crops')
        b,t = valid.shape
        if valid.dtype != torch.bool or not bool(valid.any(1).all()): raise ValueError('Each video must have a valid face frame')
        if x.shape[:2] != valid.shape or indices.shape != valid.shape: raise ValueError('Frame mask/index shape mismatch')
        flat_valid = valid.flatten(); raw = x.reshape(b*t,3,224,224)[flat_valid]
        fmap = self.backbone((raw-self.mean)/self.std)
        if fmap.shape[1:] != (1280,7,7): raise RuntimeError('Unexpected EfficientNet geometry')
        rgb_valid = self.rgb(fmap.flatten(2).transpose(1,2))
        rgb = rgb_valid.new_zeros(b*t,49,192); rgb[flat_valid] = rgb_valid
        rgb = rgb.reshape(b,t,49,192)
        forensic = rgb.new_zeros(b*t,49,32)
        if self.level >= 1:
            ff = self.fixed(raw, self.level >= 5).flatten(2).transpose(1,2)
            forensic[flat_valid] = self.forensic_proj(F.layer_norm(ff,(8,))).to(rgb.dtype)
        difference = torch.zeros_like(rgb)
        pairs = near_pairs(valid, indices)
        if self.level >= 2: difference[:,1:] = (rgb[:,1:]-rgb[:,:-1])*pairs[:,:,None,None]
        dt = self.delta(difference) if self.level >= 2 else rgb.new_zeros(b,t,49,32)
        h = self.norm(torch.cat([rgb, forensic.reshape(b,t,49,32), dt], -1))
        h = h + self.adapter(h); shared = h + self.shared(h)
        flat = shared.reshape(b*t,49,256)
        if self.level >= 4: routed, balance, stats = self.moe(flat,flat_valid)
        else:
            routed = flat; balance = flat.sum()*0
            stats = dict(requested=torch.zeros(4,device=x.device),accepted=torch.zeros(4,device=x.device),overflow=torch.tensor(0.,device=x.device),tokens=flat_valid.sum()*49,entropy=torch.tensor(0.,device=x.device))
        routed = routed.reshape(b,t,49,256)
        ml = self.m(routed).squeeze(-1); ql = self.q(routed).squeeze(-1); ul = self.u(routed).squeeze(-1)
        frame_shared = shared.mean(2)
        global_feature = (frame_shared*valid[...,None]).sum(1)/valid.sum(1,keepdim=True)
        global_logit = self.classifier(global_feature).squeeze(-1)
        if self.level >= 3:
            evidence = ml.float().sigmoid()
            if self.level >= 6: evidence = evidence*ql.float().sigmoid()*(1-ul.float().sigmoid())
            weights = evidence/evidence.sum(2,keepdim=True).clamp_min(1e-6)
            spatial = .5*routed.mean(2) + .5*(routed*weights[...,None]).sum(2)
        else: spatial = routed.mean(2)
        gate = valid.float()
        if self.level >= 2:
            adjacent = torch.zeros_like(spatial); count = spatial.new_zeros(b,t,1)
            adjacent[:,1:] += spatial[:,:-1]*pairs[...,None]; count[:,1:] += pairs[...,None]
            adjacent[:,:-1] += spatial[:,1:]*pairs[...,None]; count[:,:-1] += pairs[...,None]
            neighbors = adjacent/count.clamp_min(1)
            learned = self.temporal_gate(torch.cat([spatial,neighbors],-1).transpose(1,2)).squeeze(1).sigmoid()
            gate = gate * torch.where(count.squeeze(-1)>0, .25+.75*learned, torch.ones_like(learned))
        temporal = (spatial*gate[...,None]).sum(1)/gate.sum(1,keepdim=True).clamp_min(1e-6)
        # A2 uses temporal features in the shared classifier even before localization.
        if self.level == 2: global_logit = self.classifier(.5*global_feature+.5*temporal).squeeze(-1)
        correction = .5*torch.tanh(self.local_classifier(temporal).squeeze(-1)) if self.level >= 3 else torch.zeros_like(global_logit)
        return dict(logit=global_logit+correction, global_logit=global_logit, mask_logits=ml, quality_logits=ql, uncertainty_logits=ul, balance=balance, routing=stats)

def objectives(a, other, batch, fraction, level):
    y = batch['y'].float(); valid = batch['valid']; zero = a['logit'].sum()*0
    cls = sum(F.binary_cross_entropy_with_logits(o[k].float(),y) for o in (a,other) for k in ('logit','global_logit'))/4
    target = F.adaptive_avg_pool2d(batch['mask'].flatten(0,1)[:,None],(7,7)).reshape(*valid.shape,49)
    known = batch['known'] & valid
    def mask_loss(o):
        if not bool(known.any()): return zero
        logits = o['mask_logits'][known].float(); truth = target[known]
        bce = F.binary_cross_entropy_with_logits(logits,truth)
        fake = truth.sum(-1)>0
        dice = zero
        if bool(fake.any()):
            p = logits[fake].sigmoid(); q = truth[fake]
            dice = (1-(2*(p*q).sum(-1)+1)/(p.sum(-1)+q.sum(-1)+1)).mean()
        return bce+dice
    loc = (mask_loss(a)+mask_loss(other))/2 if level >= 3 else zero
    pairs = near_pairs(valid,batch['indices'])
    pairs &= known[:,1:] & known[:,:-1] & ((target[:,1:].mean(-1)-target[:,:-1].mean(-1)).abs()<.1)
    temporal = zero
    if level >= 3 and bool(pairs.any()):
        means = a['mask_logits'].float().sigmoid().mean(-1)
        temporal = F.smooth_l1_loss(means[:,1:][pairs],means[:,:-1][pairs])
    quality, error = zero, zero
    if level >= 6:
        stable = (1-(a['mask_logits'].float().sigmoid()-other['mask_logits'].float().sigmoid()).abs()).detach()
        quality = sum(F.binary_cross_entropy_with_logits(o['quality_logits'][valid].float(),stable[valid]) for o in (a,other))/2
        if bool(known.any()):
            error = sum(F.binary_cross_entropy_with_logits(o['uncertainty_logits'][known].float(),(o['mask_logits'][known].float().sigmoid()-target[known]).abs().detach()) for o in (a,other))/2
    brier = sum((o['logit'].float().sigmoid()-y).square().mean() for o in (a,other))/2 if level >= 6 else zero
    ramp = min(1.,max(0.,(fraction-.2)/.1)); moe_ramp = min(1.,max(0.,(fraction-.4)/.1))
    bal = (a['balance']+other['balance'])/2 if level >= 4 else zero
    loss = cls+.2*loc+.15*ramp*temporal+.1*ramp*quality+.02*moe_ramp*bal+.05*(brier+error)
    return loss, dict(classification=cls.detach(),localization=loc.detach(),temporal=temporal.detach(),quality=quality.detach(),balance=bal.detach(),brier=brier.detach(),error_proxy=error.detach())

def region_maps(output, valid):
    maps = output['mask_logits'].float().sigmoid().reshape(-1,1,7,7)
    return F.interpolate(maps,size=(224,224),mode='bilinear',align_corners=False).reshape(*valid.shape,224,224)*valid[...,None,None]


In [ ]:
def probabilities(logits, temperature=1.):
    return torch.as_tensor(logits,dtype=torch.float64).div(temperature).sigmoid().numpy()

def ece(y,p,bins=15):
    edges = np.linspace(0,1,bins+1); result = 0.
    for i in range(bins):
        keep = (p>=edges[i]) & ((p<edges[i+1]) if i<bins-1 else (p<=edges[i+1]))
        if keep.any(): result += keep.mean()*abs(y[keep].mean()-p[keep].mean())
    return float(result)

def metrics(records, calibration=None):
    y = np.array([r['label'] for r in records],dtype=int)
    p = probabilities([r['logit'] for r in records],calibration['temperature'] if calibration else 1.)
    pred = p>=.5
    tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    two = len(np.unique(y))==2
    result = dict(n=len(y),real=int((y==0).sum()),fake=int((y==1).sum()),auc=float(roc_auc_score(y,p)) if two else None,ap=float(average_precision_score(y,p)) if two else None,accuracy=float(accuracy_score(y,pred)),balanced_accuracy=float(balanced_accuracy_score(y,pred)) if two else None,precision=float(precision_score(y,pred,zero_division=0)),recall=float(recall_score(y,pred,zero_division=0)),f1=float(f1_score(y,pred,zero_division=0)),tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp),fpr=float(fp/max(1,fp+tn)),ece=ece(y,p),brier=float(np.mean((y-p)**2)),eer=None)
    if two:
        fpr,tpr,_ = roc_curve(y,p); k = np.argmin(abs(fpr-(1-tpr)))
        result['eer'] = float((fpr[k]+1-tpr[k])/2)
    if calibration:
        positive = p>=calibration['fake_threshold']; negative = p<=calibration['clear_threshold']
        accepted = positive|negative
        mistakes = (positive & (y==0)) | (negative & (y==1))
        result['selective'] = dict(coverage=float(accepted.mean()),uncertain=int((~accepted).sum()),uncertain_real=int(((~accepted)&(y==0)).sum()),uncertain_fake=int(((~accepted)&(y==1)).sum()),risk=float(mistakes[accepted].mean()) if accepted.any() else None,fake_precision=float(y[positive].mean()) if positive.any() else None,fake_recall_all=float(positive[y==1].mean()) if (y==1).any() else None,false_clear_all_fake=float(negative[y==1].mean()) if (y==1).any() else None,fake_flags=int(positive.sum()),clear_flags=int(negative.sum()),errors_all=int(mistakes.sum()))
    return result

def grouped_auc_interval(records, cfg):
    groups = sorted({r['group'] for r in records})
    lookup = {g:[r for r in records if r['group']==g] for g in groups}
    if len(groups)<2: return dict(low=None,high=None,groups=len(groups),valid_bootstraps=0)
    rng = np.random.default_rng(cfg.seed); scores = []
    for _ in range(cfg.bootstrap):
        sample = [r for g in rng.choice(groups,len(groups),replace=True) for r in lookup[g]]
        y = [r['label'] for r in sample]
        if len(set(y))==2: scores.append(roc_auc_score(y,[r['logit'] for r in sample]))
    return dict(low=float(np.quantile(scores,.025)) if scores else None,high=float(np.quantile(scores,.975)) if scores else None,groups=len(groups),valid_bootstraps=len(scores))

@torch.no_grad()
def evaluate(model, loader, condition):
    model.eval(); records = []; requested = np.zeros(4); accepted = np.zeros(4)
    overflow, token_count, entropy_sum = 0.,0.,0.
    inter, union, pred_sum, true_sum, mask_frames = 0.,0.,0.,0.,0
    elapsed, crops = 0.,0; latency = []
    if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats()
    for batch in loader:
        batch = move_batch(batch)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        start = time.perf_counter()
        with amp_context(): output = model(batch['x'],batch['valid'],batch['indices'])
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        duration = time.perf_counter()-start; elapsed += duration; latency.append(duration)
        crops += int(batch['valid'].sum())
        stats = output['routing']; n = float(stats['tokens'])
        requested += stats['requested'].cpu().numpy(); accepted += stats['accepted'].cpu().numpy()
        overflow += float(stats['overflow']); token_count += n; entropy_sum += float(stats['entropy'])*n
        for i,z in enumerate(output['logit'].float().cpu().tolist()):
            valid = batch['valid'][i]
            proxy = output['uncertainty_logits'][i][valid].float().sigmoid().mean().item() if model.level>=6 else None
            records.append(dict(video_id=batch['video_id'][i],dataset=batch['dataset'][i],method=batch['method'][i],group=batch['group'][i],label=int(batch['y'][i]),logit=z,uncertainty_proxy=proxy,frames=int(valid.sum()),condition=condition))
        known = batch['known'] & batch['valid']
        if model.level>=3 and bool(known.any()):
            pred = region_maps(output,batch['valid'])[known]>=.5; truth = batch['mask'][known]>=.5
            inter += float((pred & truth).sum()); union += float((pred | truth).sum())
            pred_sum += float(pred.sum()); true_sum += float(truth.sum()); mask_frames += int(known.sum())
    health = dict(requested=requested,accepted=accepted,requested_share=requested/max(1.,requested.sum()),accepted_share=accepted/max(1.,accepted.sum()),overflow_fraction=overflow/max(1.,token_count),entropy=entropy_sum/max(1.,token_count))
    localization = dict(known_frames=mask_frames,iou=inter/union if union else None,dice=2*inter/(pred_sum+true_sum) if pred_sum+true_sum else None,threshold=.5)
    speed = dict(scope='crop model only; excludes loading/face detection/video decode; first batch may be cold',seconds=elapsed,crops=crops,crops_per_second=crops/max(elapsed,1e-9),batch_p50_seconds=float(np.median(latency)) if latency else None,batch_p95_seconds=float(np.quantile(latency,.95)) if latency else None,peak_cuda_allocated_bytes=torch.cuda.max_memory_allocated() if DEVICE.type=='cuda' else None)
    return records,dict(routing=health,localization=localization,speed=speed)

def report_slices(records, calibration=None, bootstrap=False):
    reports = {}
    for dataset in sorted({r['dataset'] for r in records}):
        subset = [r for r in records if r['dataset']==dataset]
        reports[dataset] = metrics(subset,calibration)
        if bootstrap: reports[dataset]['auc_interval'] = grouped_auc_interval(subset,CFG)
        for method in sorted({r['method'] for r in subset if r['label']==1}):
            group = [r for r in subset if r['label']==0 or r['method']==method]
            key = dataset+'/'+method; reports[key] = metrics(group,calibration)
            if bootstrap: reports[key]['auc_interval'] = grouped_auc_interval(group,CFG)
    return reports

def validate(model, rows, cfg):
    summary, score_values = {}, []
    for condition in CONDITIONS:
        loader = make_loader(ClipDataset(rows,'dev',cfg,condition),cfg)
        records,diagnostics = evaluate(model,loader,condition)
        slices = report_slices(records)
        for key,value in slices.items():
            if value['auc'] is None: raise ValueError('Undefined development AUC: '+key)
            score_values.append(value['auc']-.25*value['ece']-.25*value['fpr'])
        summary[condition] = dict(metrics=slices,diagnostics=diagnostics)
    return min(score_values),summary

def calibrate(records, cfg, checkpoint_hash, manifest_hash):
    z = torch.tensor([r['logit'] for r in records],dtype=torch.float64)
    y = torch.tensor([r['label'] for r in records],dtype=torch.float64)
    if len(y.unique())!=2: raise ValueError('Calibration requires both labels')
    # Bounded positive scalar temperature; no test labels used.
    raw = nn.Parameter(torch.tensor(math.log((1-.05)/(20-1)),dtype=torch.float64))
    optimizer = torch.optim.LBFGS([raw],lr=.2,max_iter=100,line_search_fn='strong_wolfe')
    def closure():
        optimizer.zero_grad(); temperature=.05+19.95*raw.sigmoid()
        loss=F.binary_cross_entropy_with_logits(z/temperature,y); loss.backward(); return loss
    optimizer.step(closure)
    temperature=float((.05+19.95*raw.sigmoid()).detach())
    p=probabilities(z.numpy(),temperature); labels=y.numpy().astype(int)
    candidates=np.unique(p); fake_options=[]
    for threshold in candidates:
        positive=p>=threshold
        fpr=positive[labels==0].mean(); recall=positive[labels==1].mean()
        if fpr<=cfg.max_fpr and recall>=cfg.min_fake_recall: fake_options.append((float(recall),float(threshold)))
    result=dict(temperature=temperature,fake_threshold=1.000001,clear_threshold=-.000001,status='unvalidated_all_uncertain',scope='synthetic held-out calibration; not deployment validation' if cfg.synthetic_dev_cal else 'actual held-out calibration; domain shift still unvalidated',n_real=int((labels==0).sum()),n_fake=int((labels==1).sum()),max_fpr=cfg.max_fpr,max_false_clear=cfg.max_false_clear,min_fake_recall=cfg.min_fake_recall,checkpoint_sha256=checkpoint_hash,manifest_sha256=manifest_hash)
    if fake_options:
        best_recall=max(v[0] for v in fake_options)
        high=max(v[1] for v in fake_options if v[0]==best_recall)
        lows=[float(v) for v in candidates if v<high and ((p<=v)[labels==1].mean()<=cfg.max_false_clear) and ((p<=v)&(labels==0)).any()]
        result.update(fake_threshold=high,clear_threshold=max(lows) if lows else -.000001,status='empirical_constraints_met_not_certified')
    result['calibration_metrics']=metrics(records,result)
    return result

def file_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()
